# Tutorial 2 — Beta Weight Estimation: LSA and LSS

**Learning objectives**
1. Understand the General Linear Model (GLM) framework for fMRI
2. Explain the haemodynamic response function (HRF) and its role in design matrices
3. Implement Least Squares All (LSA) and Least Squares Separate (LSS) beta estimation
4. Diagnose design matrix collinearity and understand when LSS is preferred
5. Inspect and save per-trial beta weight maps

---


## Section 0 — Setup


In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import image, plotting
from nilearn.glm.first_level import (FirstLevelModel,
                                      make_first_level_design_matrix)
from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_design_matrix

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from fmri_helpers import (
    make_lsa_events, make_lss_events,
    fit_lsa_run, fit_lss_run,
    save_beta_manifest, track_runtime
)

SUBJECT       = "sub-1"
USER          = os.getenv("USER")
FMRIPREP_ROOT = f"/scratch/alpine/{USER}/haxby2001/derivatives/fmriprep"
BIDS_ROOT     = f"/scratch/alpine/{USER}/haxby2001/"
TR            = 2.5
RUN           = 1

BOLD_PATH   = f"{FMRIPREP_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN}_desc-hmp24_bold.nii.gz"
MASK_PATH   = f"{FMRIPREP_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN}_desc-brain_mask.nii.gz"
EVENTS_PATH = f"{BIDS_ROOT}/{SUBJECT}/func/{SUBJECT}_task-objectviewing_run-{RUN:02}_events.tsv"

bold_img   = image.load_img(BOLD_PATH)
mask_img   = image.load_img(MASK_PATH)
events_df  = pd.read_csv(EVENTS_PATH, sep='\t')

print(f"BOLD shape    : {bold_img.shape}")
print(f"Events loaded : {len(events_df)} trials  ×  {events_df['trial_type'].nunique()} conditions")
print(events_df.head())


---
## Section 1 — The General Linear Model (GLM) and HRF

The GLM models the BOLD time series at each voxel as a weighted sum of
predictor time series (regressors):

$$y = X\beta + \epsilon$$

Where:
- $y$ is the BOLD time series (n_volumes × 1)
- $X$ is the **design matrix** (n_volumes × n_regressors)
- $\beta$ are the **beta weights** (n_regressors × 1) — what we estimate
- $\epsilon$ is the residual (unexplained variance)

### The Hemodynamic Response Function (HRF)

Neural activity cannot be directly measured with fMRI. Instead, we measure the
BOLD (Blood-Oxygen-Level-Dependent) signal, which reflects changes in blood
oxygenation driven by neurovascular coupling.

The HRF describes the expected BOLD response to a brief neural event:
- Peak ~5-6 seconds after stimulus onset
- Followed by an undershoot ~15-20 seconds after onset
- Returns to baseline ~30 seconds later

Each trial's regressor is created by **convolving** a boxcar function (1 during
the stimulus, 0 otherwise) with the HRF.


In [ ]:
# Visualise the HRF (SPM canonical double-gamma)
# Use nilearn's HRF computation
from nilearn.glm.first_level.hemodynamic_models import spm_hrf
hrf = spm_hrf(TR, oversampling=16, time_length=32)
t_hrf = np.arange(len(hrf)) * TR / 16

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# HRF
axes[0].plot(t_hrf, hrf, lw=2, color='steelblue')
axes[0].axhline(0, color='gray', lw=0.8, ls=':')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('BOLD response')
axes[0].set_title('SPM canonical HRF')

# Illustrate convolution for a 1-second stimulus at t=0
stim = np.zeros(len(t_hrf)); stim[t_hrf <= 1] = 1.0
bold_pred = np.convolve(stim, hrf)[:len(t_hrf)]
axes[1].fill_between(t_hrf, stim, alpha=0.3, color='steelblue', label='Stimulus boxcar')
axes[1].plot(t_hrf, bold_pred, color='red', lw=2, label='Predicted BOLD (HRF-convolved)')
axes[1].set_xlabel('Time (s)'); axes[1].legend()
axes[1].set_title('Stimulus boxcar → convolution with HRF → predicted BOLD')

plt.tight_layout(); plt.show()


---
## Section 2 — Design Matrix Construction

### Exercise ✏️ 1 — Build a standard condition-level design matrix

Before we look at trial-wise betas, let's understand a standard condition-level
design matrix where each condition gets one HRF-convolved regressor.


In [ ]:
# YOUR CODE HERE
# Build a design matrix with ONE regressor per condition (not per trial)
# Use: make_first_level_design_matrix()
# Required arguments:
#   frame_times  — array of volume acquisition times (seconds)
#   events       — the events_df dataframe directly (no relabelling needed)
#   hrf_model    — 'spm'
#   drift_model  — 'polynomial'

n_vols      = bold_img.shape[-1]
frame_times = np.arange(n_vols) * TR   # volume onset times

dm_conditions = make_first_level_design_matrix(
    frame_times = ...,
    events      = ...,
    hrf_model   = ...,
    drift_model = ...,
)

print(f"Design matrix shape   : {dm_conditions.shape}")
print(f"Columns (conditions)  : {list(dm_conditions.columns[:8])}")

ax = plot_design_matrix(dm_conditions)
ax.set_title("Condition-level design matrix (1 regressor per condition)")
plt.tight_layout(); plt.show()


---
## Section 3 — LSA vs LSS: Two Approaches to Trial Beta Estimation

For MVPA, we need **one beta weight map per trial** — this tells us
what pattern of brain activity was present for each individual stimulus.

### Least Squares All (LSA)

All trials are modelled simultaneously in a single GLM.
Each trial gets its own unique regressor (e.g., `face__trial_0042`).

**Problem:** With short inter-trial intervals (ITIs), adjacent trial regressors
become collinear (correlated) — the GLM cannot reliably separate their estimates.

### Least Squares Separate (LSS — Mumford et al. 2012)

For each trial, a *separate* GLM is fitted with only 3 types of regressors:
1. **Target regressor**: the trial being estimated
2. **Same-condition others**: all other trials of the same condition merged
3. **Other conditions**: one regressor per other condition

This dramatically reduces collinearity for the target trial's estimate,
at the cost of running n_trials separate GLMs (parallelised with joblib).


In [ ]:
# Compare design matrix structure: LSA vs LSS for the same run

# --- LSA design events (one unique label per trial) ---
ev_lsa = make_lsa_events(events_df)
print(f"LSA: {ev_lsa['trial_type'].nunique()} unique regressors (one per trial)")
print(ev_lsa.head(3))

# --- LSS design events for trial #0 ---
ev_lss_trial0 = make_lss_events(events_df, target_idx=0)
print(f"\nLSS (trial 0): {ev_lss_trial0['trial_type'].nunique()} regressors")
print(ev_lss_trial0.head(8))


### ✏️ Exercise 2 — Compare LSA and LSS design matrices

1. Build an LSA design matrix for Run 1 using `make_lsa_events()` and `make_first_level_design_matrix()`
2. Build an LSS design matrix for trial 0 using `make_lss_events()`
3. Compute the correlation between the first two trial regressors in each matrix
4. Comment: why is LSS less affected by collinearity?


In [ ]:
# YOUR CODE HERE

# Part 1: LSA design matrix
ev_lsa = make_lsa_events(events_df)
dm_lsa = make_first_level_design_matrix(
    frame_times = ...,
    events      = ...,
    hrf_model   = 'spm',
    drift_model = 'polynomial',
)

# Part 2: LSS design matrix for trial 0
ev_lss = make_lss_events(events_df, target_idx=0)
dm_lss = make_first_level_design_matrix(...)

# Part 3: Correlation between first two trial regressors in LSA
r_lsa = np.corrcoef(dm_lsa.iloc[:, 0], dm_lsa.iloc[:, 1])[0, 1]
print(f"LSA — correlation between trial 0 and trial 1 regressors: r = {r_lsa:.3f}")

# In LSS, trial 0 is the target regressor; trial 1 is merged into "others"
# so they are NOT directly compared — comment below on what this implies:
# YOUR COMMENT:

# Visualise both design matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_design_matrix(dm_lsa, axes=None)  # will open a new figure
plt.title("LSA — all trial regressors"); plt.tight_layout(); plt.show()

ax_lss = plot_design_matrix(dm_lss)
ax_lss.set_title("LSS — design matrix for trial 0"); plt.tight_layout(); plt.show()


---
## Section 4 — Fitting the GLM and Extracting Betas

`fit_lsa_run()` and `fit_lss_run()` wrap nilearn's `FirstLevelModel` and
return a dictionary of beta images:
```
{trial_idx: {'img': NIfTI, 'condition': str, 'onset': float, 'run': int}}
```


In [ ]:
# Fit LSS betas for Run 1 (this takes a few minutes — n_trials GLMs in parallel)
print(f"Fitting LSS for {len(events_df)} trials...")
with track_runtime("LSS Run 1"):
    lss_betas_run1 = fit_lss_run(
        bold_img   = bold_img,
        events_df  = events_df,
        mask_img   = mask_img,
        tr         = TR,
        hrf_model  = 'spm',
        run_id     = RUN,
        n_jobs     = -1,
    )

print(f"\nBeta images estimated : {len(lss_betas_run1)}")
print(f"Conditions present    : {sorted(set(v['condition'] for v in lss_betas_run1.values()))}")


In [ ]:
# Inspect one beta image
trial_idx = 0
beta_info = lss_betas_run1[trial_idx]

print(f"Trial {trial_idx}:")
print(f"  Condition : {beta_info['condition']}")
print(f"  Onset     : {beta_info['onset']:.1f} s")
print(f"  Beta image: {beta_info['img'].shape}")

# Background: fMRIPrep boldref for this subject/run
BOLDREF_PATH = (f"{FMRIPREP_ROOT}/{SUBJECT}/func/"
                f"{SUBJECT}_task-objectviewing_run-{RUN}_boldref.nii.gz")

# Visualise
vec = NiftiMasker(mask_img=mask_img).fit_transform(beta_info['img']).ravel()
vmax = float(np.percentile(np.abs(vec), 95))

display = plotting.plot_stat_map(
    beta_info['img'],
    bg_img    = BOLDREF_PATH,
    display_mode = 'z', cut_coords = 5,
    colorbar  = True, cmap = 'RdBu_r',
    threshold = vmax * 0.5,   # show top 50% of the range
    vmax      = vmax,
    title     = f"LSS beta — trial {trial_idx} ({beta_info['condition']})",
)
plt.show()


### ✏️ Exercise 3 — Beta distribution by condition

Using the LSS betas from Run 1:
1. For each trial, extract the mean beta value across all in-mask voxels
2. Create a box plot showing the distribution of mean beta values per condition
3. Do the conditions show different mean activation levels?


In [ ]:
# YOUR CODE HERE

masker = NiftiMasker(mask_img=mask_img, standardize=None, detrend=False)
masker.fit()

# Collect mean beta per trial
rows = []
for trial_idx, info in lss_betas_run1.items():
    vec = masker.transform(info['img']).ravel()
    rows.append({'trial_idx': trial_idx, 'condition': info['condition'],
                 'mean_beta': vec.mean()})
beta_df = pd.DataFrame(rows)

# YOUR PLOT:
# fig, ax = plt.subplots(...)
# ...

plt.show()


---
## Section 5 — Saving Betas and the Manifest CSV

`save_beta_manifest()` writes each 3-D beta NIfTI to disk and returns a
**manifest DataFrame** with one row per trial — this is the input to the
MVPA and searchlight notebooks.


In [ ]:
OUTPUT_DIR = f"{DERIV_ROOT}/beta_weights/{SUBJECT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Convert run-level dict to combined (run_id, trial_idx) dict
betas_combined = {(RUN, k): v for k, v in lss_betas_run1.items()}

manifest_df = save_beta_manifest(
    betas_dict  = betas_combined,
    method_name = "lss",
    output_dir  = OUTPUT_DIR,
    subject_id  = SUBJECT,
)

print(manifest_df.head())
print(f"\nManifest columns: {list(manifest_df.columns)}")


---
## Summary

| Concept | Key take-away |
|---|---|
| GLM | Each voxel modelled as weighted sum of HRF-convolved trial regressors |
| HRF | Transforms neural event timing → predicted BOLD waveform |
| LSA | One GLM, all trials; suffers from collinearity with short ITIs |
| LSS | One GLM per trial; reduces collinearity at cost of compute time |
| Beta manifest | CSV index of all saved beta images — input to MVPA notebook |

---


---
## 📖 Answer Key

### Exercise 1 — Build condition-level design matrix

In [ ]:
# ANSWER KEY — Exercise 1
n_vols      = bold_img.shape[-1]
frame_times = np.arange(n_vols) * TR

dm_conditions = make_first_level_design_matrix(
    frame_times = frame_times,
    events      = events_df,       # no relabelling — one regressor per unique trial_type
    hrf_model   = 'spm',
    drift_model = 'polynomial',
)
print(f"Design matrix shape: {dm_conditions.shape}")
print(f"Conditions: {[c for c in dm_conditions.columns if not c.startswith('poly')]}")
ax = plot_design_matrix(dm_conditions)
ax.set_title("Condition-level design matrix"); plt.tight_layout(); plt.show()


### Exercise 2 — Compare LSA and LSS design matrices

In [ ]:
# ANSWER KEY — Exercise 2
ev_lsa = make_lsa_events(events_df)
dm_lsa = make_first_level_design_matrix(
    frame_times=frame_times, events=ev_lsa,
    hrf_model='spm', drift_model='polynomial')

ev_lss = make_lss_events(events_df, target_idx=0)
dm_lss = make_first_level_design_matrix(
    frame_times=frame_times, events=ev_lss,
    hrf_model='spm', drift_model='polynomial')

r_lsa = np.corrcoef(dm_lsa.iloc[:, 0], dm_lsa.iloc[:, 1])[0, 1]
print(f"LSA — correlation between trial 0 and trial 1 regressors: r = {r_lsa:.3f}")
# In LSS, trial 1 is merged into the "others" regressor for the trial-0 GLM,
# so the target regressor (trial 0) only competes with one lumped "others"
# regressor rather than individual trial-1 regressors — dramatically lower r.
print("\nIn LSS the target trial has only ~3 main regressors, so collinearity is much lower.")

ax = plot_design_matrix(dm_lsa)
ax.set_title(f"LSA — {dm_lsa.shape[1]} columns"); plt.tight_layout(); plt.show()

ax = plot_design_matrix(dm_lss)
ax.set_title(f"LSS trial 0 — {dm_lss.shape[1]} columns"); plt.tight_layout(); plt.show()


### Exercise 3 — Beta distribution by condition

In [ ]:
# ANSWER KEY — Exercise 3
masker = NiftiMasker(mask_img=mask_img, standardize=None, detrend=False); masker.fit()

rows = []
for trial_idx, info in lss_betas_run1.items():
    vec = masker.transform(info['img']).ravel()
    rows.append({'condition': info['condition'], 'mean_beta': vec.mean()})
beta_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4))
conds = sorted(beta_df['condition'].unique())
data  = [beta_df[beta_df['condition']==c]['mean_beta'].values for c in conds]
ax.boxplot(data, labels=conds, patch_artist=True)
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('Condition'); ax.set_ylabel('Mean beta (across mask voxels)')
ax.set_title('LSS beta distribution per condition — Run 1')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

# Some conditions (e.g., faces) may show higher mean activation;
# however, between-condition differences are subtle because the mask covers
# the whole brain. MVPA exploits the spatial PATTERN, not the mean level.
